# Empty String Text Embed

Create a text embedding for an empty string using the FLUX.2 Klein text encoder (Qwen3), with configurable `max_sequence_length`.

## Setup

In [ ]:
import os
import torch
from diffusers import Flux2KleinPipeline
from transformers import Qwen2TokenizerFast, Qwen3ForCausalLM

MODEL_ID = "black-forest-labs/FLUX.2-klein-base-4B"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16
TARGET_DIR = "./empty_string_embeds"  # directory to save prompt_embeds

## Load text encoder and tokenizer

In [ ]:
text_encoder = Qwen3ForCausalLM.from_pretrained(
    MODEL_ID,
    subfolder="text_encoder",
    torch_dtype=DTYPE,
).to(DEVICE).eval()
text_encoder.requires_grad_(False)

tokenizer = Qwen2TokenizerFast.from_pretrained(
    MODEL_ID,
    subfolder="tokenizer",
)

## Encode empty string with max_sequence_length

In [ ]:
max_sequence_length = 128  # or 256, 512, etc.

with torch.no_grad():
    prompt_embeds = Flux2KleinPipeline._get_qwen3_prompt_embeds(
        text_encoder=text_encoder,
        tokenizer=tokenizer,
        prompt=[""],  # empty string
        device=DEVICE,
        dtype=DTYPE,
        max_sequence_length=max_sequence_length,
        hidden_states_layers=(9, 18, 27),
    )

print("Shape:", prompt_embeds.shape)
print("Dtype:", prompt_embeds.dtype)

# Save to target path with length in filename
os.makedirs(TARGET_DIR, exist_ok=True)
save_path = os.path.join(TARGET_DIR, f"empty_string_prompt_embeds_len{max_sequence_length}.pt")
torch.save(prompt_embeds.cpu(), save_path)
print("Saved:", save_path)